# Extract Lakehouse & Warehouse Metadata
Extract metadata from **Fabric Lakehouses** and **Warehouses** for lineage tracking.

This notebook extracts:
- **Lakehouse metadata**: Tables, files, OneLake paths, SQL endpoint details
- **Warehouse metadata**: Tables, schemas, connection details
- **Workspace information**: Workspace ID, capacity details

Primary output tables:
- `lineage_lakehouses` - Lakehouse metadata
- `lineage_lakehouse_tables` - Tables within lakehouses
- `lineage_warehouses` - Warehouse metadata
- `lineage_warehouse_tables` - Tables within warehouses

## 1. Imports

Import required libraries for extraction.

In [1]:
import json
from datetime import datetime
from typing import Any, Dict, List

from pyspark.sql import SparkSession
from sempy.fabric import list_items

spark = globals().get("spark")
if spark is None:
    spark = SparkSession.builder.getOrCreate()

print("✅ Imports successful (sempy + spark)")

StatementMeta(, eca84759-f110-4508-a664-004a447292c6, 8, Finished, Available, Finished, False)

✅ Imports successful (sempy + spark)


## 2. Configuration
Set workspace IDs and write mode.

In [2]:
# Configuration for extraction
WORKSPACE_IDS = ["2b1d454b-61a1-4abb-96c3-2b476d945a96", "242dd37c-864e-4fec-af02-1caaf4d3b1a2"]
WRITE_MODE = "append"  # use "overwrite" for clean reruns

print(f"Extracting from {len(WORKSPACE_IDS)} workspace(s)")
print(f"Delta write mode: {WRITE_MODE}")

StatementMeta(, eca84759-f110-4508-a664-004a447292c6, 10, Finished, Available, Finished, False)

Extracting from 2 workspace(s)
Delta write mode: append


## 3. Helper Functions
Functions for Delta table operations and data preparation.

In [3]:
def _serialize_value(value: Any) -> Any:
    """Serialize values for Delta table storage."""
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, datetime):
        return value.isoformat()
    return json.dumps(value, default=str)


def _append_to_delta(table_name: str, rows: List[Dict[str, Any]], mode: str = "append", max_retries: int = 3) -> int:
    """Write rows to Delta table with retry logic."""
    if not rows:
        return 0

    import time
    import pandas as pd
    from pyspark.sql.utils import AnalysisException
    
    # Convert to pandas first for better schema handling
    df_pandas = pd.DataFrame(rows)
    
    # Replace None with empty strings for object columns
    for col in df_pandas.columns:
        if df_pandas[col].dtype == 'object' and df_pandas[col].isna().all():
            df_pandas[col] = df_pandas[col].fillna("")
    
    for attempt in range(max_retries):
        try:
            # Clear Spark cache before Delta operations
            spark.catalog.clearCache()
            
            # Create Spark DataFrame
            df = spark.createDataFrame(df_pandas)
            
            # Write to Delta table using saveAsTable to ensure proper catalog registration
            writer = df.write.format("delta").mode(mode).option("mergeSchema", "true")
            
            # Save as table (registers in catalog and creates Delta format)
            writer.saveAsTable(table_name)
            
            return len(rows)
            
        except Exception as e:
            error_msg = str(e)
            
            # Check if it's a temporary table error
            if "#DDM_" in error_msg or "Invalid object name" in error_msg or "Code=208" in error_msg:
                if attempt < max_retries - 1:
                    print(f"    ⚠️  Temporary table error on attempt {attempt + 1}, retrying...")
                    spark.catalog.clearCache()
                    spark.sparkContext.setLocalProperty("spark.sql.execution.id", None)
                    time.sleep(1)
                    continue
                else:
                    print(f"    ❌ Failed after {max_retries} attempts: {error_msg}")
                    raise
            else:
                print(f"    ❌ Delta write error: {error_msg}")
                raise
    
    return 0


print("✅ Helper functions loaded")

StatementMeta(, eca84759-f110-4508-a664-004a447292c6, 11, Finished, Available, Finished, False)

✅ Helper functions loaded


## 4. Lakehouse Extraction Functions
Extract lakehouse metadata including tables and files.

In [4]:
def extract_lakehouse_metadata(lakehouse_id: str, workspace_id: str, lakehouse_info: Dict[str, Any]) -> Dict[str, Any]:
    """Extract metadata from a single lakehouse."""
    lakehouse_name = lakehouse_info.get("Display Name") or lakehouse_info.get("displayName") or str(lakehouse_id)
    print(f"\n🏗️  Extracting lakehouse: {lakehouse_name}")
    
    extraction_start = datetime.now()
    extraction_timestamp = extraction_start.isoformat()
    
    try:
        # Extract tables by querying the lakehouse
        tables = []
        table_count = 0
        
        try:
            # List tables in the lakehouse using Spark
            # This assumes the lakehouse is attached or accessible
            all_tables = spark.catalog.listTables()
            
            for table in all_tables:
                if table.database and "lakehouse" in table.database.lower():
                    # Get table details
                    table_df = spark.table(f"`{table.database}`.`{table.name}`")
                    row_count = table_df.count()
                    
                    tables.append({
                        "lakehouse_id": lakehouse_id,
                        "lakehouse_name": lakehouse_name,
                        "workspace_id": workspace_id,
                        "table_name": table.name,
                        "table_type": table.tableType,
                        "database": table.database,
                        "is_temporary": table.isTemporary,
                        "row_count": row_count,
                        "column_count": len(table_df.columns),
                        "columns": json.dumps([{"name": col, "type": str(dtype)} for col, dtype in table_df.dtypes]),
                        "extraction_timestamp": extraction_timestamp
                    })
                    table_count += 1
                    
        except Exception as e:
            print(f"  ⚠️  Could not list tables: {str(e)}")
        
        extraction_end = datetime.now()
        duration = (extraction_end - extraction_start).total_seconds()
        
        print(f"  ✅ Extracted: {table_count} tables")
        
        # Prepare lakehouse row
        lakehouse_row = {
            "workspace_id": workspace_id,
            "lakehouse_id": lakehouse_id,
            "lakehouse_name": lakehouse_name,
            "display_name": lakehouse_info.get("Display Name") or lakehouse_info.get("displayName"),
            "description": lakehouse_info.get("Description") or lakehouse_info.get("description"),
            "extraction_timestamp": extraction_timestamp,
            "extraction_method": "sempy.fabric.list_items + spark.catalog",
            "extraction_duration": duration,
            "count_tables": table_count,
        }
        
        table_rows = {
            "lineage_lakehouses": [lakehouse_row],
            "lineage_lakehouse_tables": tables
        }
        
        return {
            "artifactId": lakehouse_id,
            "artifactType": "Lakehouse",
            "workspaceId": workspace_id,
            "timestamp": extraction_timestamp,
            "lakehouseName": lakehouse_name,
            "status": "success",
            "counts": {"tables": table_count},
            "tableRows": table_rows,
            "metadata": {
                "extractionDuration": duration,
                "status": "success"
            }
        }
        
    except Exception as e:
        extraction_end = datetime.now()
        duration = (extraction_end - extraction_start).total_seconds()
        error_message = str(e)
        
        print(f"  ❌ Error: {error_message}")
        
        return {
            "artifactId": lakehouse_id,
            "artifactType": "Lakehouse",
            "workspaceId": workspace_id,
            "timestamp": extraction_timestamp,
            "lakehouseName": lakehouse_name,
            "status": "error",
            "counts": {},
            "tableRows": {},
            "metadata": {
                "status": "error",
                "errorMessage": error_message,
                "extractionDuration": duration
            }
        }


print("✅ Lakehouse extraction function ready")

StatementMeta(, eca84759-f110-4508-a664-004a447292c6, 12, Finished, Available, Finished, False)

✅ Lakehouse extraction function ready


## 5. Warehouse Extraction Functions
Extract warehouse metadata including tables and schemas.

In [5]:
def extract_warehouse_metadata(warehouse_id: str, workspace_id: str, warehouse_info: Dict[str, Any]) -> Dict[str, Any]:
    """Extract metadata from a single warehouse."""
    warehouse_name = warehouse_info.get("Display Name") or warehouse_info.get("displayName") or str(warehouse_id)
    print(f"\n🏢 Extracting warehouse: {warehouse_name}")
    
    extraction_start = datetime.now()
    extraction_timestamp = extraction_start.isoformat()
    
    try:
        # Note: Warehouses require SQL endpoint connection for detailed table extraction
        # For now, we'll extract basic metadata
        # Full table extraction would require connecting to the warehouse SQL endpoint
        
        tables = []
        table_count = 0
        
        print(f"  ℹ️  Warehouse table extraction requires SQL endpoint connection")
        print(f"  ℹ️  Storing basic metadata only")
        
        extraction_end = datetime.now()
        duration = (extraction_end - extraction_start).total_seconds()
        
        print(f"  ✅ Extracted warehouse metadata")
        
        # Prepare warehouse row
        warehouse_row = {
            "workspace_id": workspace_id,
            "warehouse_id": warehouse_id,
            "warehouse_name": warehouse_name,
            "display_name": warehouse_info.get("Display Name") or warehouse_info.get("displayName"),
            "description": warehouse_info.get("Description") or warehouse_info.get("description"),
            "extraction_timestamp": extraction_timestamp,
            "extraction_method": "sempy.fabric.list_items",
            "extraction_duration": duration,
            "count_tables": table_count,
        }
        
        table_rows = {
            "lineage_warehouses": [warehouse_row],
            "lineage_warehouse_tables": tables
        }
        
        return {
            "artifactId": warehouse_id,
            "artifactType": "Warehouse",
            "workspaceId": workspace_id,
            "timestamp": extraction_timestamp,
            "warehouseName": warehouse_name,
            "status": "success",
            "counts": {"tables": table_count},
            "tableRows": table_rows,
            "metadata": {
                "extractionDuration": duration,
                "status": "success"
            }
        }
        
    except Exception as e:
        extraction_end = datetime.now()
        duration = (extraction_end - extraction_start).total_seconds()
        error_message = str(e)
        
        print(f"  ❌ Error: {error_message}")
        
        return {
            "artifactId": warehouse_id,
            "artifactType": "Warehouse",
            "workspaceId": workspace_id,
            "timestamp": extraction_timestamp,
            "warehouseName": warehouse_name,
            "status": "error",
            "counts": {},
            "tableRows": {},
            "metadata": {
                "status": "error",
                "errorMessage": error_message,
                "extractionDuration": duration
            }
        }


print("✅ Warehouse extraction function ready")

StatementMeta(, eca84759-f110-4508-a664-004a447292c6, 13, Finished, Available, Finished, False)

✅ Warehouse extraction function ready


## 6. Process All Lakehouses
Extract metadata from all lakehouses across target workspaces.

In [ ]:
# Clear Spark cache before extraction
print("🧹 Clearing Spark cache...")
spark.catalog.clearCache()
print("✅ Spark session cleaned")

# Process all lakehouses
all_lakehouse_results = []
lakehouse_table_write_counts: Dict[str, int] = {}

for workspace_id in WORKSPACE_IDS:
    print(f"\n🔍 Processing lakehouses in workspace: {workspace_id}")
    
    try:
        # List lakehouses in workspace
        lakehouses_df = list_items(item_type="Lakehouse", workspace=workspace_id)
        
        if lakehouses_df.empty:
            print("  No lakehouses found in workspace")
            continue
        
        lakehouses = lakehouses_df.to_dict("records")
        print(f"  Found {len(lakehouses)} lakehouse(s)")
        
        write_mode = WRITE_MODE
        
        for lakehouse_info in lakehouses:
            lakehouse_id = lakehouse_info.get("Id") or lakehouse_info.get("id")
            lakehouse_name = lakehouse_info.get("Display Name") or lakehouse_info.get("displayName") or lakehouse_id
            
            if not lakehouse_id:
                print(f"  ⚠️  Skipping lakehouse '{lakehouse_name}' - no valid ID")
                continue
            
            print(f"  Processing: {lakehouse_name}")
            result = extract_lakehouse_metadata(
                lakehouse_id=lakehouse_id,
                workspace_id=workspace_id,
                lakehouse_info=lakehouse_info
            )
            all_lakehouse_results.append(result)
            
            # Write to Delta tables
            for table_name, rows in result.get("tableRows", {}).items():
                written_count = _append_to_delta(table_name, rows, mode=write_mode)
                if written_count:
                    lakehouse_table_write_counts[table_name] = lakehouse_table_write_counts.get(table_name, 0) + written_count
                    print(f"    └─ Wrote {written_count} row(s) to {table_name}")
            
            # After first write, switch to append mode
            if write_mode == "overwrite":
                write_mode = "append"
    
    except Exception as e:
        print(f"  ❌ Workspace error: {str(e)}")

print(f"\n✅ Lakehouse extraction complete: {len(all_lakehouse_results)} lakehouse(s) processed")

StatementMeta(, 75f8bea0-1bcb-467f-ab4c-d48c67b01824, 5, Finished, Available, Finished, False)

🧹 Clearing Spark cache...
✅ Spark session cleaned


NameError: name 'Dict' is not defined

## 7. Process All Warehouses
Extract metadata from all warehouses across target workspaces.

In [7]:
# Process all warehouses
all_warehouse_results = []
warehouse_table_write_counts: Dict[str, int] = {}

for workspace_id in WORKSPACE_IDS:
    print(f"\n🔍 Processing warehouses in workspace: {workspace_id}")
    
    try:
        # List warehouses in workspace
        warehouses_df = list_items(item_type="Warehouse", workspace=workspace_id)
        
        if warehouses_df.empty:
            print("  No warehouses found in workspace")
            continue
        
        warehouses = warehouses_df.to_dict("records")
        print(f"  Found {len(warehouses)} warehouse(s)")
        
        write_mode = WRITE_MODE
        
        for warehouse_info in warehouses:
            warehouse_id = warehouse_info.get("Id") or warehouse_info.get("id")
            warehouse_name = warehouse_info.get("Display Name") or warehouse_info.get("displayName") or warehouse_id
            
            if not warehouse_id:
                print(f"  ⚠️  Skipping warehouse '{warehouse_name}' - no valid ID")
                continue
            
            print(f"  Processing: {warehouse_name}")
            result = extract_warehouse_metadata(
                warehouse_id=warehouse_id,
                workspace_id=workspace_id,
                warehouse_info=warehouse_info
            )
            all_warehouse_results.append(result)
            
            # Write to Delta tables
            for table_name, rows in result.get("tableRows", {}).items():
                written_count = _append_to_delta(table_name, rows, mode=write_mode)
                if written_count:
                    warehouse_table_write_counts[table_name] = warehouse_table_write_counts.get(table_name, 0) + written_count
                    print(f"    └─ Wrote {written_count} row(s) to {table_name}")
            
            # After first write, switch to append mode
            if write_mode == "overwrite":
                write_mode = "append"
    
    except Exception as e:
        print(f"  ❌ Workspace error: {str(e)}")

print(f"\n✅ Warehouse extraction complete: {len(all_warehouse_results)} warehouse(s) processed")

StatementMeta(, eca84759-f110-4508-a664-004a447292c6, 15, Finished, Available, Finished, False)


🔍 Processing warehouses in workspace: 2b1d454b-61a1-4abb-96c3-2b476d945a96
  Found 1 warehouse(s)
  Processing: Sample with data

🏢 Extracting warehouse: Sample with data
  ℹ️  Warehouse table extraction requires SQL endpoint connection
  ℹ️  Storing basic metadata only
  ✅ Extracted warehouse metadata
    ❌ Delta write error: An error occurred while calling o6828.save.
: Operation failed: "Bad Request", 400, HEAD, http://onelake.dfs.fabric.microsoft.com/a559a09d-159b-43f9-a5f7-f908bc5a77bb/user/trusted-service-user/Tables/lineage_warehouses/_delta_log?upn=false&action=getStatus&timeout=90
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.completeExecute(AbfsRestOperation.java:231)
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.lambda$execute$0(AbfsRestOperation.java:191)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDurationOfInvocation(IOStatisticsBinding.java:464)
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.execute(AbfsR

## 8. Summary
Display extraction results for lakehouses and warehouses.

In [8]:
# Display combined summary
all_results = all_lakehouse_results + all_warehouse_results
success_count = sum(1 for r in all_results if r.get("status") == "success")
error_count = len(all_results) - success_count

print("\n" + "="*70)
print("EXTRACTION SUMMARY (LAKEHOUSES + WAREHOUSES)")
print("="*70)

print(f"\n📊 LAKEHOUSES")
print(f"Total Lakehouses: {len(all_lakehouse_results)}")
lakehouse_success = sum(1 for r in all_lakehouse_results if r.get("status") == "success")
print(f"✅ Successful: {lakehouse_success}")
print(f"❌ Errors: {len(all_lakehouse_results) - lakehouse_success}")
if lakehouse_table_write_counts:
    print("\nDelta tables written:")
    for table_name in sorted(lakehouse_table_write_counts.keys()):
        print(f"  - {table_name}: {lakehouse_table_write_counts[table_name]} row(s)")

print(f"\n🏢 WAREHOUSES")
print(f"Total Warehouses: {len(all_warehouse_results)}")
warehouse_success = sum(1 for r in all_warehouse_results if r.get("status") == "success")
print(f"✅ Successful: {warehouse_success}")
print(f"❌ Errors: {len(all_warehouse_results) - warehouse_success}")
if warehouse_table_write_counts:
    print("\nDelta tables written:")
    for table_name in sorted(warehouse_table_write_counts.keys()):
        print(f"  - {table_name}: {warehouse_table_write_counts[table_name]} row(s)")

print("\n" + "="*70)
print(f"OVERALL: {success_count}/{len(all_results)} successful extractions")
print("="*70)

# Display detailed counts
if all_lakehouse_results:
    total_tables = sum(r.get("counts", {}).get("tables", 0) for r in all_lakehouse_results if r.get("status") == "success")
    print(f"\n📈 Total lakehouse tables extracted: {total_tables}")

StatementMeta(, eca84759-f110-4508-a664-004a447292c6, 16, Finished, Available, Finished, False)


EXTRACTION SUMMARY (LAKEHOUSES + WAREHOUSES)

📊 LAKEHOUSES
Total Lakehouses: 1
✅ Successful: 1
❌ Errors: 0

🏢 WAREHOUSES
Total Warehouses: 2
✅ Successful: 2
❌ Errors: 0

OVERALL: 3/3 successful extractions

📈 Total lakehouse tables extracted: 0
